# Deep Agents 研究代理

使用 create_deep_agent 建立研究助理。TodoListMiddleware 提供規劃工具。內建檔案工具管理虛擬檔案。lookup_reference 提供本地摘要。

學習目標：載入設定、建立代理、觀察結果，再測試 HTTP 服務。

官方範例使用網路搜尋。本實驗使用本地摘要，讓你只需設定 LiteLLM 金鑰。保留規劃與檔案工具。虛擬檔案存放在單次執行的狀態中。

## 1. 準備環境

從本資料夾執行 `uv sync --locked --group notebook`。
再執行 `uv run --group notebook jupyter lab tutorial.ipynb`。

In [1]:
import sys
from pathlib import Path

assert sys.version_info[:2] == (3, 13), "請選擇 Python 3.13 核心。"
assert Path("agent.py").is_file(), "請從本專案資料夾啟動 Notebook。"
RUN_LIVE = False
print("Python 版本：", sys.version.split()[0])

Python 版本： 3.13.15


## 2. 檢查設定

設定來自環境變數或 `.env`。此步驟不輸出金鑰。
離線學習不需要有效設定。真實推論需要模型別名。

In [3]:
from settings import ConfigurationError, Settings

try:
    settings = Settings.load()
    print("模型：", settings.model)
    print("時間限制：", settings.timeout)
except ConfigurationError as exc:
    settings = None
    print(str(exc))

模型： gpt-4.1-mini
時間限制： 120.0


## 3. 先測試資料工具

此工具只讀取本地摘要。它不是網路搜尋。

In [4]:
from agent import lookup_reference

print(lookup_reference.invoke({"topic": "langgraph"}))
print(lookup_reference.invoke({"topic": "deepagents"}))

LangGraph 以節點與邊控制工作流程。狀態會傳遞至下一個節點。
Deep Agents 建立在 LangGraph 上。它提供規劃、虛擬檔案與子代理工具。


## 4. 建立 Deep Agent

`create_deep_agent` 加入檔案工具。本實驗另以 `TodoListMiddleware` 加入 `write_todos`。
預設 backend 把檔案存放在執行狀態中。
`/report.md` 不會寫入主機根目錄。

In [5]:
from agent import build_agent
from model import create_model

if settings is not None:
    graph = build_agent(create_model(settings))
    print("已建立代理。此步驟尚未呼叫 LLM。")
else:
    print("設定完成後，可在此建立代理。")

已建立代理。此步驟尚未呼叫 LLM。


## 5. 執行真實任務

確認 `.env` 的端點、金鑰與模型名稱。
將第一個程式區塊中的 `RUN_LIVE` 改為 `True`。
重新執行設定步驟，再執行下列區塊。這會使用模型額度。

In [ ]:
from agent import run_agent

result = None
RUN_LIVE = True
if RUN_LIVE:
    result = await run_agent(
        "比較 langgraph 與 deepagents。先規劃，再查詢本地摘要。將結論寫入 /report.md。"
    )
    print(result["output"])
else:
    print("已略過真實推論。將 RUN_LIVE 改為 True 可執行。")

## 6. 觀察結果

檢查工具結果或代理輸出。每次執行都是獨立任務。

In [ ]:
if result is not None:
    print(result.get("todos", []))
    print(result.get("files", {}))
else:
    print("尚未執行真實任務。")

## 7. 測試 Docker 服務

在終端機執行 `docker compose up --build -d --wait`。
開啟 `http://localhost:8002/docs`。
先呼叫 `/health`，再呼叫 `/invoke`。
完成後執行 `docker compose down`。
若 LiteLLM 只監聽 Linux 主機的 `127.0.0.1`，請改用下列命令：

```bash
docker compose -f compose.yaml -f docker/compose.host.yaml up --build -d --wait
```

停止時使用 `docker compose -f compose.yaml -f docker/compose.host.yaml down`。


In [ ]:
RUN_HTTP = False
if RUN_HTTP:
    import httpx

    async with httpx.AsyncClient(timeout=180) as client:
        response = await client.post(
            "http://localhost:8002/invoke",
            json={
                "prompt": "比較 langgraph 與 deepagents。先規劃，再查詢本地摘要。將結論寫入 /report.md。"
            },
        )
        response.raise_for_status()
        print(response.json())

## 8. 延伸練習

新增一個本地摘要。請代理比較三個主題。再檢查 todos 與 files。

## 官方來源

- [Deep Agents 官方 Quickstart](https://docs.langchain.com/oss/python/deepagents/quickstart)
- [Deep Agents 官方 Backend 文件](https://docs.langchain.com/oss/python/deepagents/backends)